# Setup - Installing Required Libraries

1. transformers — HuggingFace's library, used to load the pretrained models (both the image classifier and the audio classifier)
2. torch (PyTorch) — the deep learning engine that runs underneath these models
3. torchvision — helper functions for image processing, required alongside torch for image models
4. opencv-python (imported as `cv2`) — used to extract frames from the video
5. moviepy — used to extract the audio track out of the video file
6. pillow (imported as `PIL`) — used to format/handle images before feeding them to the model

In [1]:
import cv2
import os
from transformers import pipeline
from PIL import Image
from moviepy.editor import VideoFileClip

d:\New folder (2)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ModuleNotFoundError: No module named 'moviepy.editor'

# Checking Dependencies

In [ ]:
import cv2
from transformers import pipeline
from moviepy.editor import VideoFileClip
print("All imports are working fine!")

All imports are working fine!


## Video Branch - Frame-by-Frame Deepfake Check

This function reads the input video frame-by-frame using OpenCV, samples every 15th frame 
(checking all frames would be slow and unnecessary), and passes each sampled frame through 
a pretrained Vision Transformer (ViT) model that classifies it as "Real" or "Deepfake" with 
a confidence score. Results for all checked frames are collected into a list.

#### Face Detection Filter - Only Analyze Frames With a Face

Since our deepfake model is trained specifically on faces, we now use OpenCV's built-in 
face detector (Haar Cascade - no extra install needed) to check if a face actually exists 
in the frame before running it through the classifier. Frames with no detected face are 
skipped entirely, which avoids the confused/flickering predictions we saw on the screen 
recording test.

In [ ]:
import cv2
# Haar Cascade OpenCV ke saath hi aata hai, kuch alag install karne ki zaroorat nahi
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def analyze_video(video_path, every_n_frames=15):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    results = []
    while True:
        success, frame = cap.read()
        if not success:
            break
        if frame_count % every_n_frames == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

            if len(faces) > 0:  # sirf tab check karo jab face mila ho
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_image = Image.fromarray(frame_rgb)
                prediction = image_classifier(pil_image)
                results.append({"frame": frame_count, "prediction": prediction})
        frame_count += 1
    cap.release()
    return results

### Working OR Not?

##### 1. Access Check

In [ ]:
import os

video_path = "/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences/01_02__exit_phone_room__YVGY8LOK.mp4"  # apna actual path daal

print("File exists?", os.path.exists(video_path))

cap = cv2.VideoCapture(video_path)
print("Video opened?", cap.isOpened())
print("Total frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

File exists? True
Video opened? True
Total frames: 210.0


In [ ]:
'''
------ Changing the processing from GPU to CPU ------
'''

image_classifier = pipeline("image-classification", model="prithivMLmods/Deep-Fake-Detector-v2-Model", device=-1)

config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


2. Analysis Test

In [ ]:
test_results = analyze_video("/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences/01_03__talking_against_wall__JZUXXFRB.mp4")
print(test_results)

[{'frame': 0, 'prediction': [{'label': 'Deepfake', 'score': 0.737598180770874}, {'label': 'Realism', 'score': 0.262401819229126}]}, {'frame': 15, 'prediction': [{'label': 'Deepfake', 'score': 0.7334667444229126}, {'label': 'Realism', 'score': 0.2665331959724426}]}, {'frame': 30, 'prediction': [{'label': 'Deepfake', 'score': 0.7370829582214355}, {'label': 'Realism', 'score': 0.26291701197624207}]}, {'frame': 45, 'prediction': [{'label': 'Deepfake', 'score': 0.7482081651687622}, {'label': 'Realism', 'score': 0.2517918050289154}]}, {'frame': 60, 'prediction': [{'label': 'Deepfake', 'score': 0.7326959371566772}, {'label': 'Realism', 'score': 0.26730409264564514}]}, {'frame': 75, 'prediction': [{'label': 'Deepfake', 'score': 0.7381761074066162}, {'label': 'Realism', 'score': 0.2618238627910614}]}, {'frame': 90, 'prediction': [{'label': 'Deepfake', 'score': 0.7420881986618042}, {'label': 'Realism', 'score': 0.2579118609428406}]}, {'frame': 105, 'prediction': [{'label': 'Deepfake', 'score': 0

## Audio Branch - Voice Deepfake Check

This function extracts the audio track from the video using MoviePy and saves it as a .wav 
file. The audio is then passed through a pretrained transformer-based model 
(`MelodyMachine/Deepfake-audio-detection-V2`) which classifies the voice as "Real" or "Fake" 
with a confidence score - catching AI-generated/cloned voice artifacts that aren't visible 
in the video frames alone. Running on CPU (device=-1) to avoid the earlier CUDA compatibility issue.

In [ ]:
def analyze_audio(video_path):
    video = VideoFileClip(video_path)

    if video.audio is None:
        print("Audio not found in this video.")
        return None

    audio_path = "extracted_audio.wav"
    video.audio.write_audiofile(audio_path, logger=None)

    prediction = audio_classifier(audio_path)
    return prediction

## Standalone Audio-Only Analysis Function

For cases where the user only has an audio file (not a full video), this function skips 
video processing entirely and passes the audio file straight to the audio deepfake classifier.

In [ ]:
def analyze_audio_direct(audio_path):
    prediction = audio_classifier(audio_path)
    return prediction

## Testing the Audio Branch

Running the audio analysis function on the same test video to check if the audio classifier 
loads correctly and returns a Real/Fake prediction with confidence scores.

In [ ]:
audio_result = analyze_audio("/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences/02__talking_angry_couch.mp4")
print(audio_result)

Audio not found in this video.
None


## Fusion Layer - Combining Video + Audio Results

This function runs both the video and audio branches on the same input. It averages the 
"fake" confidence across all checked video frames, and - if audio exists - factors in the 
audio model's fake score too. If no audio track was found, the final verdict relies on the 
video branch alone. This is the "Analyzer" part of the project: it doesn't just say Real/Fake, 
it also lists which specific frames were flagged as suspicious, giving explainability.

In [ ]:
def full_analysis(video_path):
    video_results = analyze_video(video_path)
    audio_result = analyze_audio(video_path)

    # Video frames ka average fake score nikaalo
    fake_scores = []
    for r in video_results:
        for pred in r["prediction"]:
            if pred["label"].lower() in ["fake", "deepfake"]:
                fake_scores.append(pred["score"])

    avg_video_fake_score = sum(fake_scores) / len(fake_scores) if fake_scores else 0

    # Audio score check karo - agar audio mila hi nahi, toh 0 rakho aur note kar do
    audio_fake_score = 0
    audio_available = audio_result is not None

    if audio_available:
        for pred in audio_result:
            if pred["label"].lower() == "fake":
                audio_fake_score = pred["score"]

    # Agar audio nahi hai, sirf video score se decide karo; warna dono ka average
    if audio_available:
        final_score = (avg_video_fake_score + audio_fake_score) / 2
    else:
        final_score = avg_video_fake_score

    final_verdict = "FAKE" if final_score > 0.5 else "REAL"

    return {
        "video_fake_score": round(avg_video_fake_score, 3),
        "audio_fake_score": round(audio_fake_score, 3) if audio_available else "N/A (no audio track)",
        "final_verdict": final_verdict,
        "flagged_frames": [
    r["frame"] for r in video_results 
    if r["prediction"][0]["label"].lower() in ["fake", "deepfake"] 
    and r["prediction"][0]["score"] > 0.65]
    }

## Testing the Full Pipeline

Running the combined analysis end-to-end on the test video to get a final verdict, 
per-modality scores, and the list of flagged frames.

In [ ]:
result = full_analysis("/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences/01_02__walking_outside_cafe_disgusted__YVGY8LOK.mp4")
print(result)

Audio not found in this video.
{'video_fake_score': 0.757, 'audio_fake_score': 'N/A (no audio track)', 'final_verdict': 'FAKE', 'flagged_frames': [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180, 195, 210]}


# Local Setup - Installing Streamlit + Same Backend Libraries

Since Streamlit runs as a local web app (not inside a notebook), we install the same 
libraries here on the laptop, plus Streamlit itself which turns our Python script into 
a working webpage.

In [ ]:
pip install streamlit transformers torch torchvision opencv-python moviepy pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 98.2 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 5.3.1
    Uninstalling decorator-5.3.1:
      Successfully uninstalled decorator-5.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mne 1.12.1 requires decorator>=5.1, but you have decorator 4.4.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Full Streamlit App - DeepFake Analyzer Demo

This script combines everything built so far (video branch, audio branch, fusion logic) 
into a Streamlit web app. The user uploads a video, clicks "Analyze", and sees the final 
verdict along with per-modality confidence scores and flagged frames - all in a browser.

In [ ]:
import streamlit as st
import cv2
import os
from transformers import pipeline
from PIL import Image
from moviepy.editor import VideoFileClip

st.set_page_config(page_title="DeepFake Analyzer", layout="centered")

# Models ek hi baar load honge (Streamlit cache karta hai, baar baar reload nahi hota)
@st.cache_resource
def load_models():
    image_classifier = pipeline("image-classification", model="prithivMLmods/Deep-Fake-Detector-v2-Model", device=-1)
    audio_classifier = pipeline("audio-classification", model="MelodyMachine/Deepfake-audio-detection-V2", device=-1)
    return image_classifier, audio_classifier

image_classifier, audio_classifier = load_models()


def analyze_video(video_path, every_n_frames=15):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    results = []
    while True:
        success, frame = cap.read()
        if not success:
            break
        if frame_count % every_n_frames == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            prediction = image_classifier(pil_image)
            results.append({"frame": frame_count, "prediction": prediction})
        frame_count += 1
    cap.release()
    return results


def analyze_audio(video_path):
    video = VideoFileClip(video_path)
    if video.audio is None:
        return None
    audio_path = "extracted_audio.wav"
    video.audio.write_audiofile(audio_path, logger=None)
    prediction = audio_classifier(audio_path)
    return prediction


def full_analysis(video_path):
    video_results = analyze_video(video_path)
    audio_result = analyze_audio(video_path)

    fake_scores = []
    for r in video_results:
        for pred in r["prediction"]:
            if pred["label"].lower() in ["fake", "deepfake"]:
                fake_scores.append(pred["score"])
    avg_video_fake_score = sum(fake_scores) / len(fake_scores) if fake_scores else 0

    audio_fake_score = 0
    audio_available = audio_result is not None
    if audio_available:
        for pred in audio_result:
            if pred["label"].lower() == "fake":
                audio_fake_score = pred["score"]

    final_score = (avg_video_fake_score + audio_fake_score) / 2 if audio_available else avg_video_fake_score
    final_verdict = "FAKE" if final_score > 0.5 else "REAL"

    return {
        "video_fake_score": round(avg_video_fake_score, 3),
        "audio_fake_score": round(audio_fake_score, 3) if audio_available else "N/A (no audio track)",
        "final_verdict": final_verdict,
        "flagged_frames": [r["frame"] for r in video_results if r["prediction"][0]["label"].lower() in ["fake", "deepfake"]]
    }

'''
# ---------- UI shuru yahan se ----------
st.title("🔍 DeepFake Analyzer")
st.write("Upload a video to check if it's Real or a Deepfake.")

uploaded_file = st.file_uploader("Upload video", type=["mp4", "mov", "avi"])

if uploaded_file is not None:
    temp_path = "temp_video.mp4"
    with open(temp_path, "wb") as f:
        f.write(uploaded_file.read())

    st.video(temp_path)

    if st.button("Analyze"):
        with st.spinner("Analyzing... this may take a minute"):
            result = full_analysis(temp_path)

        st.subheader(f"Verdict: {result['final_verdict']}")
        st.write(f"**Video fake score:** {result['video_fake_score']}")
        st.write(f"**Audio fake score:** {result['audio_fake_score']}")
        st.write(f"**Flagged frames:** {result['flagged_frames']}")

'''
'''
## Updated UI — Let User Choose Video or Audio-Only Upload

We add a radio button so the user can choose whether they're uploading a full video 
(runs both video + audio branches) or just an audio file (runs only the audio branch). 
This makes the tool useful even when someone only has a voice clip to check.
'''
# ---------- UI shuru yahan se ---------- #
st.title("🔍 DeepFake Analyzer")
st.write("Upload a video or an audio file to check if it's Real or a Deepfake.")

input_type = st.radio("What are you uploading?", ["Video", "Audio only"])

if input_type == "Video":
    uploaded_file = st.file_uploader("Upload video", type=["mp4", "mov", "avi"])

    if uploaded_file is not None:
        temp_path = "temp_video.mp4"
        with open(temp_path, "wb") as f:
            f.write(uploaded_file.read())

        st.video(temp_path)

        if st.button("Analyze"):
            with st.spinner("Analyzing... this may take a minute"):
                result = full_analysis(temp_path)

            st.subheader(f"Verdict: {result['final_verdict']}")
            st.write(f"**Video fake score:** {result['video_fake_score']}")
            st.write(f"**Audio fake score:** {result['audio_fake_score']}")
            st.write(f"**Flagged frames:** {result['flagged_frames']}")

else:  # Audio only
    uploaded_audio = st.file_uploader("Upload audio", type=["wav", "mp3"])

    if uploaded_audio is not None:
        temp_audio_path = "temp_audio.wav"
        with open(temp_audio_path, "wb") as f:
            f.write(uploaded_audio.read())

        st.audio(temp_audio_path)

        if st.button("Analyze Audio"):
            with st.spinner("Analyzing audio..."):
                prediction = analyze_audio_direct(temp_audio_path)

            top_pred = max(prediction, key=lambda x: x["score"])
            verdict = "FAKE" if top_pred["label"].lower() == "fake" else "REAL"

            st.subheader(f"Verdict: {verdict}")
            st.write(f"**Confidence:** {round(top_pred['score'], 3)}")
            st.write(f"**Full breakdown:** {prediction}")

2026-08-17 15:29:24.139 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 15:29:24.142 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

2026-08-17 15:29:30.141 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 15:29:30.342 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-17 15:29:30.343 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 15:29:30.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 15:29:30.345 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 15:29:30.345 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 15:29:30.346 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 15:29:30.347 Thread 'MainThread': mi

### Verification - Working of Face-Detection

In [ ]:
import cv2
from transformers import pipeline
from PIL import Image

image_classifier = pipeline("image-classification", model="prithivMLmods/Deep-Fake-Detector-v2-Model", device=-1)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def analyze_video(video_path, every_n_frames=15):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    results = []
    while True:
        success, frame = cap.read()
        if not success:
            break
        if frame_count % every_n_frames == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
            if len(faces) > 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_image = Image.fromarray(frame_rgb)
                prediction = image_classifier(pil_image)
                results.append({"frame": frame_count, "prediction": prediction})
        frame_count += 1
    cap.release()
    return results

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
test = analyze_video("/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences/01__kitchen_pan.mp4")
print(test)

[{'frame': 0, 'prediction': [{'label': 'Deepfake', 'score': 0.6928364038467407}, {'label': 'Realism', 'score': 0.30716362595558167}]}, {'frame': 15, 'prediction': [{'label': 'Deepfake', 'score': 0.8143294453620911}, {'label': 'Realism', 'score': 0.18567058444023132}]}, {'frame': 30, 'prediction': [{'label': 'Deepfake', 'score': 0.754153311252594}, {'label': 'Realism', 'score': 0.2458467036485672}]}, {'frame': 45, 'prediction': [{'label': 'Deepfake', 'score': 0.7974990606307983}, {'label': 'Realism', 'score': 0.20250095427036285}]}, {'frame': 60, 'prediction': [{'label': 'Deepfake', 'score': 0.7907708883285522}, {'label': 'Realism', 'score': 0.20922915637493134}]}, {'frame': 75, 'prediction': [{'label': 'Deepfake', 'score': 0.8206357955932617}, {'label': 'Realism', 'score': 0.1793641895055771}]}, {'frame': 90, 'prediction': [{'label': 'Deepfake', 'score': 0.8161255121231079}, {'label': 'Realism', 'score': 0.18387450277805328}]}, {'frame': 105, 'prediction': [{'label': 'Deepfake', 'score